In [22]:
conn = sqlite3.connect("atp.db")
cur = conn.cursor()

cur.execute("DROP TABLE IF EXISTS match_stats;")
cur.execute("DROP TABLE IF EXISTS matches;")
cur.execute("DROP TABLE IF EXISTS players;")
cur.execute("DROP TABLE IF EXISTS tournaments;")

conn.commit()
conn.close()


In [23]:
conn = sqlite3.connect("atp.db")
cur = conn.cursor()
cur.execute("SELECT name FROM sqlite_master WHERE type='table';")
cur.fetchall()


[('sqlite_sequence',)]

In [25]:
import sqlite3

conn = sqlite3.connect("atp.db")
cur = conn.cursor()

cur.executescript("""

PRAGMA foreign_keys = ON;

DROP TABLE IF EXISTS match_stats;
DROP TABLE IF EXISTS matches;
DROP TABLE IF EXISTS players;
DROP TABLE IF EXISTS tournaments;

CREATE TABLE tournaments (
    tourney_id TEXT PRIMARY KEY,
    tourney_name TEXT NOT NULL,
    surface TEXT,
    draw_size INTEGER,
    tourney_level TEXT NOT NULL,
    tourney_date TEXT NOT NULL,
    CHECK (surface IN ('Hard','Clay','Grass','Carpet') OR surface IS NULL),
    CHECK (tourney_level IN ('G','M','A','C','F','D'))
);

CREATE TABLE players (
    player_id TEXT PRIMARY KEY,
    player_name TEXT NOT NULL,
    hand TEXT,
    ht INTEGER,
    ioc TEXT,
    CHECK (hand IN ('R','L','U') OR hand IS NULL),
    CHECK (ht > 0 OR ht IS NULL),
    CHECK (length(ioc) = 3 OR ioc IS NULL)
);

CREATE TABLE matches (
    match_id INTEGER PRIMARY KEY AUTOINCREMENT,
    tourney_id TEXT NOT NULL,
    match_num INTEGER,
    winner_id TEXT NOT NULL,
    loser_id TEXT NOT NULL,
    score TEXT,
    best_of INTEGER,
    round TEXT NOT NULL,
    minutes INTEGER,
    winner_age REAL,
    winner_rank INTEGER,
    winner_rank_points INTEGER,
    loser_age REAL,
    loser_rank INTEGER,
    loser_rank_points INTEGER,
    CHECK (best_of IN (3,5) OR best_of IS NULL),
    CHECK (minutes >= 0 OR minutes IS NULL),
    CHECK (winner_age >= 0 OR winner_age IS NULL),
    CHECK (winner_rank >= 1 OR winner_rank IS NULL),
    CHECK (winner_rank_points >= 0 OR winner_rank_points IS NULL),
    CHECK (loser_age >= 0 OR loser_age IS NULL),
    CHECK (loser_rank >= 1 OR loser_rank IS NULL),
    CHECK (loser_rank_points >= 0 OR loser_rank_points IS NULL),
    FOREIGN KEY (tourney_id) REFERENCES tournaments(tourney_id),
    FOREIGN KEY (winner_id) REFERENCES players(player_id),
    FOREIGN KEY (loser_id) REFERENCES players(player_id)
);

CREATE TABLE match_stats (
    match_id INTEGER PRIMARY KEY,
    w_ace INTEGER,
    w_df INTEGER,
    w_svpt INTEGER,
    w_1stIn INTEGER,
    w_1stWon INTEGER,
    w_2ndWon INTEGER,
    w_SvGms INTEGER,
    w_bpSaved INTEGER,
    w_bpFaced INTEGER,
    l_ace INTEGER,
    l_df INTEGER,
    l_svpt INTEGER,
    l_1stIn INTEGER,
    l_1stWon INTEGER,
    l_2ndWon INTEGER,
    l_SvGms INTEGER,
    l_bpSaved INTEGER,
    l_bpFaced INTEGER,
    FOREIGN KEY (match_id) REFERENCES matches(match_id)
);

""")

conn.commit()
conn.close()


In [26]:
import pandas as pd

# Load CSVs
tournaments = pd.read_csv("C:/Users/mrobe/Courses/Database/tournaments.csv",
                          encoding='latin1', sep=';', quotechar='"')

players = pd.read_csv("C:/Users/mrobe/Courses/Database/players.csv",
                      encoding='latin1', sep=';', quotechar='"')

matches = pd.read_csv("C:/Users/mrobe/Courses/Database/matches.csv",
                      encoding='latin1', sep=';', quotechar='"')

match_stats = pd.read_csv("C:/Users/mrobe/Courses/Database/match_stats.csv",
                          encoding='latin1', sep=';', quotechar='"')

# Clean players height
players['ht'] = players['ht'].replace(['0', '0.0', '', ' ', 'NULL'], None)
players['ht'] = pd.to_numeric(players['ht'], errors='coerce')
players.loc[players['ht'] == 0, 'ht'] = None

# Clean rank columns
rank_cols = ['winner_rank', 'winner_rank_points', 'loser_rank', 'loser_rank_points']
for col in rank_cols:
    matches[col] = matches[col].replace(['0', '0.0', '', ' ', 'NULL'], None)
    matches[col] = pd.to_numeric(matches[col], errors='coerce')
    matches.loc[matches[col] <= 0, col] = None


In [27]:
conn = sqlite3.connect("atp.db")

tournaments.to_sql("tournaments", conn, if_exists="append", index=False)
players.to_sql("players", conn, if_exists="append", index=False)
matches.to_sql("matches", conn, if_exists="append", index=False)
match_stats.to_sql("match_stats", conn, if_exists="append", index=False)

conn.close()


In [28]:
conn = sqlite3.connect("atp.db")
cur = conn.cursor()

for table in ["tournaments", "players", "matches", "match_stats"]:
    cur.execute(f"SELECT COUNT(*) FROM {table};")
    print(table, cur.fetchone()[0])

conn.close()


tournaments 2391
players 1994
matches 50179
match_stats 50179


In [29]:
conn = sqlite3.connect("atp.db")
cur = conn.cursor()

cur.execute("PRAGMA foreign_key_check;")
cur.fetchall()


[]

In [31]:
import sqlite3

conn = sqlite3.connect("atp.db")
cur = conn.cursor()

query = """
SELECT substr(tourney_id, 1, 4) AS year, COUNT(*)
FROM matches
GROUP BY year
ORDER BY year;
"""

cur.execute(query)
rows = cur.fetchall()

for row in rows:
    print(row)

conn.close()


('2000', 3364)
('2001', 3307)
('2002', 3236)
('2003', 3214)
('2004', 3277)
('2005', 3257)
('2006', 3257)
('2007', 3152)
('2008', 3110)
('2009', 3074)
('2010', 3058)
('2011', 3030)
('2012', 3025)
('2013', 2959)
('2014', 2901)
('2015', 2958)


In [32]:
def run_sql(query):
    conn = sqlite3.connect("atp.db")
    cur = conn.cursor()
    cur.execute(query)
    rows = cur.fetchall()
    conn.close()
    return rows
run_sql("""
SELECT surface, COUNT(*)
FROM tournaments
GROUP BY surface;
""")


[(None, 31), ('Carpet', 189), ('Clay', 881), ('Grass', 137), ('Hard', 1153)]

In [33]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("atp.db")

df = pd.read_sql_query("""
SELECT tourney_name, draw_size
FROM tournaments
ORDER BY draw_size DESC
LIMIT 10;
""", conn)

conn.close()

df


,tourney_name,draw_size
0,Roland Garros,128
1,Wimbledon,128
2,US Open,128
3,Australian Open,128
4,Roland Garros,128
5,Wimbledon,128
6,US Open,128
7,Australian Open,128
8,Roland Garros,128
9,Wimbledon,128


In [34]:
pd.read_sql_query("""
SELECT surface, COUNT(*) AS tournaments
FROM tournaments
GROUP BY surface;
""", sqlite3.connect("atp.db"))


,surface,tournaments
0,None,31
1,Carpet,189
2,Clay,881
3,Grass,137
4,Hard,1153
